# Milestone 3 — Frame classification v2 (marker-token pooling)

Same input as v1 (`… <t> gave </t> …`), but the frame representation is the
concatenation of the **<t> / </t> marker tokens' hidden states** instead of
[CLS] — focusing the classifier on the predicate. This targets the discrimination
gap that the soft-mask sweep (+0.002) and candidate-name conditioning (which
*hurt*, −0.047) couldn't close.

**Baseline to beat:** 0.887 (v1 marker-less model: 0.863).

> `Runtime → GPU` (A100; L4 fine with `batch_size=8`), fresh runtime, `Run all`.

In [ ]:
!nvidia-smi

In [ ]:
# --- Clone the private project repo + install pinned env --------------------
import os
try:
    from google.colab import userdata
    os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN") or ""
except Exception:
    os.environ.setdefault("GH_TOKEN", "")

![ -d Texture_Frames ] || git clone -q https://$GH_TOKEN@github.com/texturejc/Texture_Frames.git
!cd Texture_Frames && git fetch -q origin && git reset --hard -q origin/main && echo "project at $(git rev-parse --short HEAD)"
!pip install -q -r Texture_Frames/requirements-colab.txt

In [ ]:
import numpy, numpy.strings, torch, transformers
print("numpy", numpy.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import os, sys
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
sys.path.insert(0, "/content/Texture_Frames/encoder_parser")

import nltk
for pkg in ["framenet_v17", "wordnet", "omw-1.4"]:
    nltk.download(pkg)

## Sanity check — the marker tokens are found at the right positions

In [ ]:
import frame2_data
from lexicon import Lexicon
from transformers import AutoTokenizer
from data import TRIGGER_START, TRIGGER_END, load_frame_examples, mark_trigger

lex = Lexicon()
tok = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")
tok.add_special_tokens({"additional_special_tokens": [TRIGGER_START, TRIGGER_END]})
start_id = tok.convert_tokens_to_ids(TRIGGER_START)
end_id = tok.convert_tokens_to_ids(TRIGGER_END)

text, loc, gold = load_frame_examples("dev")[0]
marked = mark_trigger(text, loc)
enc = tok(marked, truncation=True, max_length=320)
sp, ep = frame2_data.find_marker_positions(enc["input_ids"], start_id, end_id)
toks = tok.convert_ids_to_tokens(enc["input_ids"])
print("input      :", marked)
print("gold frame :", gold)
print(f"markers    : start@{sp}={toks[sp]!r}  end@{ep}={toks[ep]!r}")

## Train

`OUTPUT_DIR` is local disk (Drive quota). ~20–30 min on A100.

In [ ]:
import train_frame2

OUTPUT_DIR = "/content/outputs/frame2"   # local disk — bypasses the full Drive
print("saving to (local, ephemeral):", OUTPUT_DIR)

model, tokenizer, lexicon = train_frame2.train(
    base_model="microsoft/deberta-v3-large",
    output_dir=OUTPUT_DIR,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=320,
)

## Evaluate — candidate-mask sweep (pick on dev, report on test)

In [ ]:
from eval_frame2 import evaluate_frame2, print_report

print("### DEV — pick the winning bias ###")
print_report(evaluate_frame2(model, tokenizer, lexicon, split="dev", max_length=320))

print("\n### TEST — report the DEV-chosen bias ###")
print_report(evaluate_frame2(model, tokenizer, lexicon, split="test", max_length=320))

## Interpreting

Report **frame acc vs 0.887** (and vs the v1 marker-less 0.863). If marker
pooling closes the gap, all three tasks are at/above baseline and the parser is
~4× faster — the full goal. If it lands ~0.86–0.87 like v1, marker pooling wasn't
the lever and we bank frame as competitive.